# Day 2: Chat Completions API and Open-Source Models

The Chat Completions API was invented by OpenAI and became the standard. Gemini and Ollama both expose
OpenAI-compatible endpoints, so one client library works for all of them.

In [ ]:
import os
import requests
from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI
from scraper import fetch_website_contents

load_dotenv(override=True)
api_key = os.getenv("GEMINI_API_KEY")

if not api_key:
    print("No API key was found. Add GEMINI_API_KEY to your .env file.")
elif not api_key.startswith(("AIz", "AQ.")):
    print("An API key was found, but it doesn't start with AIz or AQ.")
elif api_key.strip() != api_key:
    print("The API key has leading or trailing whitespace.")
else:
    print("API key found and looks good so far!")

## Calling the endpoint directly with HTTP

In [ ]:
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
MODEL = "gemini-3.1-flash-lite"

headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}

payload = {
    "model": MODEL,
    "messages": [{"role": "user", "content": "Tell me a fun fact"}],
}

response = requests.post(GEMINI_BASE_URL + "chat/completions", headers=headers, json=payload)
response.json()["choices"][0]["message"]["content"]

## The same call with the OpenAI client library

The `openai` package is just a lightweight wrapper around that HTTP call. Changing `base_url`
and `api_key` points it at Gemini instead of OpenAI; no OpenAI model is involved.

In [ ]:
gemini = OpenAI(base_url=GEMINI_BASE_URL, api_key=api_key)

response = gemini.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Tell me a fun fact"}],
)
response.choices[0].message.content

## Ollama running locally

Same client again, pointed at the local Ollama server.
If the next cell doesn't return `Ollama is running`, run `ollama serve` in a terminal.

In [ ]:
requests.get("http://localhost:11434").content

In [ ]:
!ollama pull llama3.2
!ollama pull deepseek-r1:1.5b

In [ ]:
OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")

for model in ["llama3.2", "deepseek-r1:1.5b"]:
    response = ollama.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": "Tell me a fun fact"}],
    )
    display(Markdown(f"**{model}**\n\n{response.choices[0].message.content}"))

## Homework: Day 1 summarizer on a local model

No API charges, and the data never leaves the machine.

In [ ]:
OLLAMA_MODEL = "llama3.2"

system_prompt = """
You are a snarky assistant that analyzes the contents of a website,
and provides a short, snarky, humorous summary, ignoring text that might be navigation related.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""

user_prompt_prefix = """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, then summarize these too.
"""


def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + website},
    ]


def summarize(url):
    website = fetch_website_contents(url)
    response = ollama.chat.completions.create(model=OLLAMA_MODEL, messages=messages_for(website))
    return response.choices[0].message.content


def display_summary(url):
    display(Markdown(summarize(url)))

In [ ]:
display_summary("https://edwarddonner.com")